In [ ]:
# 06 - ResNet18 Transfer Learning
# Model construction comes from src/models/transfer_model.py

import sys
sys.path.insert(0, '..')

import os
import yaml
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

from src.models.transfer_model import build_transfer_model
from src.data.split_dataset import check_dataset
from src.utils.metrics import (
    get_predictions,
    show_confusion_matrix,
    show_classification_report,
    plot_confusion_matrix,
)

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

In [8]:
# check device - use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [9]:
# make sure all three folders exist before going further
train_dir = "../data/processed/train"
val_dir = "../data/processed/val"
test_dir = "../data/processed/test"

for folder in [train_dir, val_dir, test_dir]:
    if not os.path.isdir(folder):
        print(folder, "not found, run 01_data_prep.ipynb first")

In [10]:
# image size 224 - ResNet18 was pretrained on ImageNet at 224x224,
# so we match that resolution instead of the CNN's 128
image_size = 224

# ImageNet normalization is REQUIRED when using ImageNet-pretrained weights -
# the pretrained features were learned on inputs normalized this exact way
transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# load all three datasets - ImageFolder uses folder names as class labels
train_data = datasets.ImageFolder(train_dir, transform=transform)
val_data = datasets.ImageFolder(val_dir, transform=transform)
test_data = datasets.ImageFolder(test_dir, transform=transform)

num_classes = len(train_data.classes)
class_names = train_data.classes

print("Classes:", train_data.class_to_idx)
print("Train size:", len(train_data), "Val size:", len(val_data), "Test size:", len(test_data))

Classes: {'healthy': 0, 'low_tread': 1, 'sidewall_damaged': 2, 'uneven_wear': 3, 'zero_tread': 4}
Train size: 520 Val size: 60 Test size: 67


In [11]:
# quick class balance check on train set - should be roughly equal after augmentation
counts = check_dataset(train_dir)

Dataset check:
healthy : 104 images 
low_tread : 104 images 
sidewall_damaged : 104 images 
uneven_wear : 104 images 
zero_tread : 104 images 
Total: 520 images


In [12]:
# grid search - try a few combinations, keep the best one based on val accuracy
# NOTE: transfer learning uses a SMALLER learning-rate range than the from-scratch CNN -
# the pretrained features shouldn't be shaken up too hard (see RULEBOOK Section 6)

learning_rates = [0.001, 0.0003]
batch_sizes = [16, 32]

best_val_acc = 0
best_settings = None

for lr in learning_rates:
    for bs in batch_sizes:
        print("Trying lr:", lr, "batch_size:", bs)

        train_loader = DataLoader(train_data, batch_size=bs, shuffle=True)
        val_loader = DataLoader(val_data, batch_size=bs, shuffle=False)

        # fresh model each trial - backbone frozen, only the new classifier head trains
        model = build_transfer_model("resnet18", num_classes, freeze_backbone=True).to(device)
        loss_function = nn.CrossEntropyLoss()

        # only optimize the parameters that actually require grad (the new fc layer)
        trainable_params = [p for p in model.parameters() if p.requires_grad]
        optimizer = optim.Adam(trainable_params, lr=lr)

        # short run per combination - just enough to compare
        for epoch in range(5):
            model.train()
            for images, labels in train_loader:
                images = images.to(device)
                labels = labels.to(device)
                optimizer.zero_grad()
                outputs = model(images)
                loss = loss_function(outputs, labels)
                loss.backward()
                optimizer.step()

        # check val accuracy after these 5 epochs
        model.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)
                outputs = model(images)
                predicted = outputs.argmax(dim=1)
                val_correct += (predicted == labels).sum().item()
                val_total += labels.size(0)

        val_acc = val_correct / val_total
        print("  val acc:", round(val_acc, 3))

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_settings = {"learning_rate": lr, "batch_size": bs}

print()
print("Best settings:", best_settings, "with val acc:", round(best_val_acc, 3))

Trying lr: 0.001 batch_size: 16
  val acc: 0.6
Trying lr: 0.001 batch_size: 32
  val acc: 0.533
Trying lr: 0.0003 batch_size: 16
  val acc: 0.483
Trying lr: 0.0003 batch_size: 32
  val acc: 0.383

Best settings: {'learning_rate': 0.001, 'batch_size': 16} with val acc: 0.6


In [13]:
# save the winning settings to the config file automatically
# this closes the loop - the config file always matches what actually won the search

config = {
    "model": "resnet18",
    "freeze_backbone": True,
    "image_size": image_size,
    "batch_size": best_settings["batch_size"],
    "epochs": 20,           # fewer than the CNN's 40 - pretrained features converge faster
    "learning_rate": best_settings["learning_rate"]
}

with open('../configs/resnet18_config.yaml', 'w') as f:
    yaml.dump(config, f)

print("Saved to configs/resnet18_config.yaml:", config)

Saved to configs/resnet18_config.yaml: {'model': 'resnet18', 'freeze_backbone': True, 'image_size': 224, 'batch_size': 16, 'epochs': 20, 'learning_rate': 0.001}


In [14]:
# load settings back from the config file
# from here on, the config file is the single source of truth - nothing is hardcoded

with open('../configs/resnet18_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

batch_size = config['batch_size']
epochs = config['epochs']
learning_rate = config['learning_rate']

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

print("Training with:", config)

Training with: {'batch_size': 16, 'epochs': 20, 'freeze_backbone': True, 'image_size': 224, 'learning_rate': 0.001, 'model': 'resnet18'}


In [15]:
# build the final model - fresh weights, not reused from the grid search
model = build_transfer_model(
    config["model"], num_classes, freeze_backbone=config["freeze_backbone"]
).to(device)

loss_function = nn.CrossEntropyLoss()

# only optimize parameters that require grad (the new classifier head when frozen)
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.Adam(trainable_params, lr=learning_rate)

print("Trainable parameters:", sum(p.numel() for p in trainable_params))

Trainable parameters: 2565


In [16]:
# final training loop with early stopping on val-accuracy improvement
# saves the best version of the model whenever val acc improves
# (this is what the commented block in 02_cnn_baseline was intending - it matters
# more here because transfer learning val curves are noisier)

os.makedirs("../results/resnet18", exist_ok=True)

patience = 5
epochs_without_improvement = 0
best_val_acc = 0
best_model_path = "../results/resnet18/model.pt"

for epoch in range(epochs):

    # training
    model.train()
    train_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        predicted = outputs.argmax(dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total

    # validation
    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            predicted = outputs.argmax(dim=1)
            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total

    print("Epoch", epoch + 1, "- train loss:", round(train_loss, 3),
          "train acc:", round(train_acc, 3), "val acc:", round(val_acc, 3))

    # check if this is the best val accuracy so far
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        epochs_without_improvement = 0
        torch.save(model.state_dict(), best_model_path)
        print("  new best val acc, model saved")
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= patience:
        print("Training stopped early - no val improvement for", patience, "epochs")
        break

print("Best val accuracy:", round(best_val_acc, 3))

Epoch 1 - train loss: 46.763 train acc: 0.398 val acc: 0.417
  new best val acc, model saved
Epoch 2 - train loss: 36.425 train acc: 0.565 val acc: 0.417
Epoch 3 - train loss: 31.522 train acc: 0.638 val acc: 0.5
  new best val acc, model saved
Epoch 4 - train loss: 29.452 train acc: 0.679 val acc: 0.533
  new best val acc, model saved
Epoch 5 - train loss: 25.957 train acc: 0.737 val acc: 0.517
Epoch 6 - train loss: 24.547 train acc: 0.754 val acc: 0.55
  new best val acc, model saved
Epoch 7 - train loss: 22.475 train acc: 0.779 val acc: 0.6
  new best val acc, model saved
Epoch 8 - train loss: 21.971 train acc: 0.794 val acc: 0.55
Epoch 9 - train loss: 20.047 train acc: 0.831 val acc: 0.567
Epoch 10 - train loss: 19.972 train acc: 0.804 val acc: 0.55
Epoch 11 - train loss: 18.691 train acc: 0.821 val acc: 0.583
Epoch 12 - train loss: 17.332 train acc: 0.831 val acc: 0.533
Training stopped early - no val improvement for 5 epochs
Best val accuracy: 0.6


In [17]:
# reload the best saved weights before evaluating
# (the model in memory is from the last epoch trained, not necessarily the best one)
model.load_state_dict(torch.load(best_model_path))
print("Loaded best saved weights for evaluation")

C:\Users\User\AppData\Local\Temp\ipykernel_3332\2193235084.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_model_path))


Loaded best saved weights for evaluation


In [18]:
# check performance on val set - this is fine to look at during development
true_labels, predicted_labels = get_predictions(model, val_loader, device)

print("VAL SET RESULTS")
show_confusion_matrix(true_labels, predicted_labels, class_names)
show_classification_report(true_labels, predicted_labels, class_names)

VAL SET RESULTS
Confusion matrix
(rows = actual, columns = predicted)
Classes: ['healthy', 'low_tread', 'sidewall_damaged', 'uneven_wear', 'zero_tread']
[[13  5  1  1  2]
 [ 1  6  0  0  4]
 [ 2  1  5  0  0]
 [ 0  2  0  6  0]
 [ 0  3  1  1  6]]
                  precision    recall  f1-score   support

         healthy       0.81      0.59      0.68        22
       low_tread       0.35      0.55      0.43        11
sidewall_damaged       0.71      0.62      0.67         8
     uneven_wear       0.75      0.75      0.75         8
      zero_tread       0.50      0.55      0.52        11

        accuracy                           0.60        60
       macro avg       0.63      0.61      0.61        60
    weighted avg       0.65      0.60      0.61        60



In [19]:
# check performance on TEST set - this is the final, honest, one-time score
# only run this once you are fully done tuning - do not go back and tune more after seeing this

test_true_labels, test_predicted_labels = get_predictions(model, test_loader, device)

print("TEST SET RESULTS (final)")
show_confusion_matrix(test_true_labels, test_predicted_labels, class_names)
show_classification_report(test_true_labels, test_predicted_labels, class_names)

TEST SET RESULTS (final)
Confusion matrix
(rows = actual, columns = predicted)
Classes: ['healthy', 'low_tread', 'sidewall_damaged', 'uneven_wear', 'zero_tread']
[[16  5  2  0  0]
 [ 0  5  0  1  6]
 [ 2  0  7  0  0]
 [ 0  1  0  5  4]
 [ 0  2  1  3  7]]
                  precision    recall  f1-score   support

         healthy       0.89      0.70      0.78        23
       low_tread       0.38      0.42      0.40        12
sidewall_damaged       0.70      0.78      0.74         9
     uneven_wear       0.56      0.50      0.53        10
      zero_tread       0.41      0.54      0.47        13

        accuracy                           0.60        67
       macro avg       0.59      0.59      0.58        67
    weighted avg       0.63      0.60      0.61        67



In [20]:
# save the test results to a text file - small file, goes in git, this is the evidence trail
from sklearn.metrics import classification_report

report_text = classification_report(test_true_labels, test_predicted_labels, target_names=class_names)

os.makedirs("../results/resnet18", exist_ok=True)

with open("../results/resnet18/metrics.txt", "w") as f:
    f.write("Settings used: " + str(config) + "\n\n")
    f.write("Test set results:\n")
    f.write(report_text)

print("Saved to results/resnet18/metrics.txt")

Saved to results/resnet18/metrics.txt


In [21]:
# save the trained model - this file is too large for git, goes to Drive instead (see RULEBOOK.txt)
torch.save(model.state_dict(), "../results/resnet18/model.pt")
print("Model saved to results/resnet18/model.pt")

Model saved to results/resnet18/model.pt


In [ ]:


def plot_confusion_matrix(
    true_labels,
    predicted_labels,
    class_names,
    title="Confusion matrix",
    ax=None,
    normalize=False,
    cmap="Blues",
):
    """
    Plot a confusion matrix with matplotlib/seaborn.
    normalize=True shows row-normalized values (per-class recall).
    """
    cm = confusion_matrix(true_labels, predicted_labels)

    fmt = "d"
    if normalize:
        cm = cm.astype("float") / cm.sum(axis=1, keepdims=True)
        fmt = ".2f"

    show_fig = ax is None
    if show_fig:
        fig, ax = plt.subplots(figsize=(7, 6))

    sns.heatmap(
        cm, annot=True, fmt=fmt, cmap=cmap,
        xticklabels=class_names, yticklabels=class_names,
        ax=ax, cbar=False,
        vmin=0 if normalize else None,
        vmax=1 if normalize else None,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(title)
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    plt.setp(ax.get_yticklabels(), rotation=0)

    if show_fig:
        plt.tight_layout()
        plt.show()

ModuleNotFoundError: No module named 'src'